# Learned re-scoring of the fused boxes (learnable model weights)

WBF averages the members' scores with weights we choose (`ensemble_weights.ipynb`: size-aware, p = 2, val mAP 0.3027). Here the weights are **learned**: a small model takes every fused box and predicts how good it is from what each member said about it.

**Features per fused box** (the box coordinates stay those of the weighted fusion; only the score changes):
- each member's score for this box: its highest-scoring same-class box with IoU ≥ 0.55 (0 if it has none);
- the same member scores split by object size (small < 32², medium < 96², large) — so the model learns **one weight per member per size**;
- the fused score, class, box area and aspect ratio, frame width.

**Target:** the IoU of the box with the ground-truth object it matches (greedy by score, same class; 0 if none). mAP averages IoU thresholds 0.5–0.95, so a box that fits its object better should rank higher. Boxes mostly inside ignored regions are left out of training.

**Models:** logistic regression (the learned weights are readable) and a small MLP (64 hidden units).

**Validation:** leave-one-clip-out over the 7 val clips: train on 6, re-score the 7th, repeat; the mAP is computed on all labelled frames with these out-of-fold scores. The final model is trained on all 7 clips and applied to the test set.

**Run after** `ensemble_weights.ipynb` (it needs `runs/ensemble/weights_best.json`, `fused_best_test.pkl` and the member caches). CPU only, about 10–15 min. Output: `runs/ensemble/submissions/L_learned.csv`.

## Setup

In [1]:
import json
import pickle
import sys
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch

SRC = Path.cwd().parent            # this notebook lives in src/experiments/
ROOT = SRC.parent
sys.path.insert(0, str(SRC))
from common.evaluate import evaluate_detections
from common.pipeline import fuse
from common.submission import write_submission

DATA_ROOT = ROOT / "multiple-object-video-detection-hw-2-ucu-ai-2026"
GT_FILE = ROOT / "data_processed" / "val" / "gt_eval_coco.json"
TEST_CSV = DATA_ROOT / "test_images.csv"
ENS = ROOT / "runs" / "ensemble"
COLS = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR"]
torch.manual_seed(0)

best = json.loads((ENS / "weights_best.json").read_text())
MEMBERS, SCHEME, P, WBF_IOU = best["members"], best["scheme"], best["p"], best["iou"]
assert not best.get("vote"), "this notebook expects a fusion without the Faster R-CNN vote"
print("fusion:", best["label"])

gt = json.loads(GT_FILE.read_text())
val_meta = {im["id"]: im for im in json.loads((DATA_ROOT / "val" / "annotations_coco.json").read_text())["images"]}
LABELLED = sorted(im["id"] for im in gt["images"])
CLIP = {i: val_meta[i]["video"] for i in LABELLED}
SHAPE = {i: (val_meta[i]["height"], val_meta[i]["width"]) for i in LABELLED}
gt_boxes, ignore = defaultdict(list), defaultdict(list)
for a in gt["annotations"]:
    (ignore if a["iscrowd"] else gt_boxes)[a["image_id"]].append(a)
val_preds = {n: pickle.loads((ENS / f"valall_{n}.pkl").read_bytes()) for n in MEMBERS}
print(f"{len(MEMBERS)} members, {len(LABELLED)} labelled val frames in {len(set(CLIP.values()))} clips")

fusion: B7+Ret | size p=2
8 members, 359 labelled val frames in 7 clips


## 1. The same weighted fusion as `ensemble_weights.ipynb`, on the labelled val frames

In [2]:
SIZES = ["AP_small", "AP_medium", "AP_large"]


def size_bin(xyxy):
    area = (xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1])
    return np.digitize(area, [32 ** 2, 96 ** 2])


def to_dets(boxes_by_id, ids):
    dets = []
    for i in ids:
        xyxy, s, c = boxes_by_id[i]
        dets += [{"image_id": i, "category_id": int(k) + 1, "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                  "score": float(v)} for (x1, y1, x2, y2), v, k in zip(xyxy, s, c)]
    return dets


# Member weights exactly as in ensemble_weights.ipynb (from each member's val AP)
member_ap = pd.DataFrame({n: evaluate_detections(GT_FILE, to_dets(val_preds[n], LABELLED), image_ids=LABELLED) for n in MEMBERS}).T
if SCHEME == "size":
    rel = (member_ap[SIZES] / member_ap[SIZES].mean()).fillna(1.0)
    W = {n: np.maximum(rel.loc[n].to_numpy(), 0.05) ** P for n in MEMBERS}
elif SCHEME == "global":
    W = {n: np.full(3, (member_ap.loc[n, "mAP"] / member_ap["mAP"].mean()) ** P) for n in MEMBERS}
else:
    W = {n: np.ones(3) for n in MEMBERS}


def fuse_weighted(preds, image_id, shape):
    per = []
    for n in MEMBERS:
        xyxy, s, c = preds[n][image_id]
        per.append((xyxy, s * W[n][size_bin(xyxy)], c))
    return fuse(per, shape, iou=WBF_IOU)


t0 = time.time()
fused_val = {i: fuse_weighted(val_preds, i, SHAPE[i]) for i in LABELLED}
base = evaluate_detections(GT_FILE, to_dets(fused_val, LABELLED), image_ids=LABELLED)
print(f"weighted fusion on val: mAP {base['mAP']:.4f} ({time.time() - t0:.0f} s; ensemble_weights.ipynb reported 0.3027)")

weighted fusion on val: mAP 0.3027 (48 s; ensemble_weights.ipynb reported 0.3027)


## 2. Features and targets

In [3]:
def box_iou(a, b):
    iw = (np.minimum(a[:, None, 2], b[None, :, 2]) - np.maximum(a[:, None, 0], b[None, :, 0])).clip(0)
    ih = (np.minimum(a[:, None, 3], b[None, :, 3]) - np.maximum(a[:, None, 1], b[None, :, 1])).clip(0)
    inter = iw * ih
    area = lambda x: (x[:, 2] - x[:, 0]) * (x[:, 3] - x[:, 1])
    return inter / (area(a)[:, None] + area(b)[None] - inter + 1e-9)


def features(fused, members, width):
    # One row per fused box: member scores, member scores x size, fused score, class, size, aspect, frame width
    xyxy, s, c = fused
    n = len(xyxy)
    ms = np.zeros((n, len(members)), np.float32)
    for m, (mx, mscore, mc) in enumerate(members):
        for k in np.unique(c):
            a, b = np.where(c == k)[0], np.where(mc == k)[0]
            if len(b):
                iou = box_iou(xyxy[a], mx[b])
                ms[a, m] = np.where(iou >= 0.55, mscore[b][None], 0).max(1)
    sb = size_bin(xyxy)
    by_size = np.concatenate([ms * (sb == k)[:, None] for k in range(3)], 1)
    w, h = xyxy[:, 2] - xyxy[:, 0], xyxy[:, 3] - xyxy[:, 1]
    other = np.stack([s, np.log(w * h + 1), np.log((w + 1) / (h + 1)), np.full(n, width / 1920)], 1)
    return np.concatenate([ms, by_size, other, np.eye(8, dtype=np.float32)[c], np.eye(3, dtype=np.float32)[sb]], 1).astype(np.float32)


FEATURE_NAMES = (MEMBERS + [f"{m} x {z}" for z in ("small", "medium", "large") for m in MEMBERS]
                 + ["fused score", "log area", "log aspect", "frame width"] + [f"class {k}" for k in range(8)] + ["small", "medium", "large"])


def targets(fused, image_id):
    # IoU with the matched ground truth (greedy by score, same class), 0 if none; keep=False inside ignored regions
    xyxy, s, c = fused
    g = gt_boxes[image_id]
    y, keep = np.zeros(len(xyxy), np.float32), np.ones(len(xyxy), bool)
    if g:
        gb = np.array([a["bbox"] for a in g], float); gb[:, 2:] += gb[:, :2]
        gc = np.array([a["category_id"] - 1 for a in g])
        iou = box_iou(xyxy, gb)
        iou[c[:, None] != gc[None]] = 0
        used = np.zeros(len(g), bool)
        for i in np.argsort(-s):
            cand = np.where(~used & (iou[i] >= 0.5))[0]
            if len(cand):
                j = cand[iou[i, cand].argmax()]
                used[j] = True
                y[i] = iou[i, j]
    ig = ignore[image_id]
    if ig:
        ib = np.array([a["bbox"] for a in ig], float); ib[:, 2:] += ib[:, :2]
        iw = (np.minimum(xyxy[:, None, 2], ib[None, :, 2]) - np.maximum(xyxy[:, None, 0], ib[None, :, 0])).clip(0)
        ih = (np.minimum(xyxy[:, None, 3], ib[None, :, 3]) - np.maximum(xyxy[:, None, 1], ib[None, :, 1])).clip(0)
        ioa = (iw * ih).max(1) / ((xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1]) + 1e-9)
        keep &= ~((y == 0) & (ioa >= 0.5))
    return y, keep


X, Y, KEEP, IDS = [], [], [], []
for i in LABELLED:
    f = fused_val[i]
    X.append(features(f, [val_preds[n][i] for n in MEMBERS], SHAPE[i][1]))
    y, k = targets(f, i)
    Y.append(y); KEEP.append(k); IDS.append(np.full(len(y), i))
X, Y, KEEP, IDS = map(np.concatenate, (X, Y, KEEP, IDS))
GROUP = np.array([CLIP[i] for i in IDS])
print(f"{len(X)} fused boxes, {X.shape[1]} features, {(Y > 0).mean():.1%} matched to an object, {(~KEEP).mean():.1%} in ignored regions")

179500 fused boxes, 47 features, 6.9% matched to an object, 10.7% in ignored regions


## 3. Leave-one-clip-out: does the learned score beat the weighted fusion?

In [4]:
def train(Xtr, ytr, kind, epochs=400):
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6
    xt, yt = torch.from_numpy((Xtr - mu) / sd), torch.from_numpy(ytr)
    if kind == "logistic":
        net = torch.nn.Linear(Xtr.shape[1], 1)
    else:
        net = torch.nn.Sequential(torch.nn.Linear(Xtr.shape[1], 64), torch.nn.ReLU(), torch.nn.Linear(64, 1))
    opt = torch.optim.Adam(net.parameters(), lr=0.01, weight_decay=1e-4)
    loss_fn = torch.nn.BCEWithLogitsLoss()      # soft targets: the IoU with the matched object
    for _ in range(epochs):
        opt.zero_grad()
        loss = loss_fn(net(xt).squeeze(1), yt)
        loss.backward()
        opt.step()
    return net, mu, sd


def predict(model, Xs):
    net, mu, sd = model
    with torch.no_grad():
        return torch.sigmoid(net(torch.from_numpy((Xs - mu) / sd)).squeeze(1)).numpy()


def rescored(scores):
    out, k = {}, 0
    for i in LABELLED:
        xyxy, _, c = fused_val[i]
        out[i] = (xyxy, scores[k:k + len(xyxy)], c)
        k += len(xyxy)
    return out


rows = {"weighted fusion (no learning)": base}
for kind in ("logistic", "mlp"):
    t0 = time.time()
    oof = np.zeros(len(X), np.float32)
    for clip in sorted(set(GROUP)):
        tr = (GROUP != clip) & KEEP
        model = train(X[tr], Y[tr], kind)
        oof[GROUP == clip] = predict(model, X[GROUP == clip])
    rows[f"learned: {kind} (out-of-fold)"] = evaluate_detections(GT_FILE, to_dets(rescored(oof), LABELLED), image_ids=LABELLED)
    print(f"{kind}: mAP {rows[f'learned: {kind} (out-of-fold)']['mAP']:.4f} ({time.time() - t0:.0f} s)", flush=True)
table = pd.DataFrame(rows).T[COLS]
table["delta mAP"] = table["mAP"] - base["mAP"]
table.round(4)

logistic: mAP 0.2790 (302 s)
mlp: mAP 0.2400 (598 s)


,mAP,AP50,AP75,AP_small,AP_medium,AP_large,AR,delta mAP
weighted fusion (no learning),0.3027,0.5553,0.2954,0.1476,0.2593,0.4986,0.5216,0.0000
learned: logistic (out-of-fold),0.2790,0.5174,0.2728,0.1453,0.2489,0.4624,0.5216,-0.0237
learned: mlp (out-of-fold),0.2400,0.4602,0.2264,0.1346,0.2313,0.3890,0.5216,-0.0626


In [5]:
# Per clip, the best learned model vs the weighted fusion: is the gain consistent?
BEST_KIND = "logistic" if rows["learned: logistic (out-of-fold)"]["mAP"] >= rows["learned: mlp (out-of-fold)"]["mAP"] else "mlp"
oof = np.zeros(len(X), np.float32)
for clip in sorted(set(GROUP)):
    tr = (GROUP != clip) & KEEP
    oof[GROUP == clip] = predict(train(X[tr], Y[tr], BEST_KIND), X[GROUP == clip])
learned_val = rescored(oof)
per_clip = []
for clip in sorted(set(GROUP)):
    ids = [i for i in LABELLED if CLIP[i] == clip]
    per_clip.append({"clip": clip, "weighted": evaluate_detections(GT_FILE, to_dets(fused_val, ids), image_ids=ids)["mAP"],
                     f"learned ({BEST_KIND})": evaluate_detections(GT_FILE, to_dets(learned_val, ids), image_ids=ids)["mAP"]})
per_clip = pd.DataFrame(per_clip)
per_clip["delta"] = per_clip.iloc[:, 2] - per_clip["weighted"]
print(f"learned better on {(per_clip.delta > 0).sum()} of {len(per_clip)} clips")
per_clip.round(4)

learned better on 4 of 7 clips


,clip,weighted,learned (logistic),delta
0,uav0000086_00000_v,0.3157,0.3182,0.0025
1,uav0000117_02622_v,0.3166,0.2935,-0.0232
2,uav0000137_00458_v,0.3460,0.3462,0.0003
3,uav0000182_00000_v,0.2370,0.2293,-0.0077
4,uav0000268_05773_v,0.2000,0.2008,0.0007
5,uav0000305_00000_v,0.4270,0.4180,-0.0090
6,uav0000339_00001_v,0.3146,0.3155,0.0009


## 4. The learned weights

Logistic regression trained on all 7 clips: the weight of each member's score, overall and per object size (features are standardised, so weights are comparable). A larger weight means that member's opinion counts more for whether a box is right.

In [6]:
final_logistic = train(X[KEEP], Y[KEEP], "logistic")
w = final_logistic[0].weight.detach().numpy().ravel()
M = len(MEMBERS)
weights = pd.DataFrame({"overall": w[:M], "small": w[M:2 * M], "medium": w[2 * M:3 * M], "large": w[3 * M:4 * M]}, index=MEMBERS)
weights.round(3)

,overall,small,medium,large
yolo_low_scale_960,-0.108,0.041,0.039,0.052
yolo_baseline_1280,-0.071,0.073,0.114,0.035
yolo_final_960,-0.084,0.056,-0.028,-0.089
yolo_final_1280,0.100,-0.093,-0.057,-0.040
rtdetr_960,0.080,-0.042,-0.059,-0.012
rtdetr_final_960,0.029,-0.017,0.068,0.038
rtdetr_final_960_s025,0.046,-0.032,-0.005,0.094
retinanet_baseline_ms,0.050,0.003,0.005,-0.007


**Observations:** _fill in_

## 5. Test submission

The chosen model (`BEST_KIND`), trained on all 7 val clips, re-scores the weighted fusion's test boxes (`fused_best_test.pkl` from `ensemble_weights.ipynb`). Box coordinates are unchanged.

In [ ]:
final = final_logistic if BEST_KIND == "logistic" else train(X[KEEP], Y[KEEP], "mlp")
test = pd.read_csv(TEST_CSV)
width = dict(zip(test["image_id"], test["width"]))
test_preds = {n: pickle.loads((ENS / f"test_{n}.pkl").read_bytes()) for n in MEMBERS}
fused_dets = pickle.loads((ENS / "fused_best_test.pkl").read_bytes())
by_img = defaultdict(list)
for d in fused_dets:
    by_img[d["image_id"]].append(d)
t0 = time.time()
rows_by_id = {}
for image_id, ds in by_img.items():
    xyxy = np.array([[x, y, x + w_, y + h] for x, y, w_, h in (d["bbox"] for d in ds)], np.float32)
    s = np.array([d["score"] for d in ds], np.float32)
    c = np.array([d["category_id"] - 1 for d in ds])
    new = predict(final, features((xyxy, s, c), [test_preds[n][image_id] for n in MEMBERS], width[image_id]))
    rows_by_id[image_id] = [(int(k), float(v), float(b[0]), float(b[1]), float(b[2] - b[0]), float(b[3] - b[1]))
                            for b, v, k in zip(xyxy, new, c)]
sub = write_submission(rows_by_id, TEST_CSV, ENS / "submissions" / "L_learned.csv")
n_boxes = sub.PredictionString.fillna("").str.split().str.len().fillna(0) // 6
assert len(sub) == len(test) and n_boxes.max() <= 500
print(f"wrote L_learned.csv ({BEST_KIND}): {len(sub)} rows, {int((n_boxes > 0).sum())} with detections ({time.time() - t0:.0f} s)")